# 06 pycavi SWarp

This notebooks run SWarp over the images for which Scamp creates a .head file. <br>
Tested with SWarp 2.38.0.

Uses the default configuration for SWarp, in case of needs, additional configuration parameters can be changed to optimize the performance in Chapman for example

In [ ]:
import os
import sys
import glob
import logging
from astropy.io import fits

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production

from pycavi import pycavi_tools
from pycavi.pycavi_astromatic_tools import get_swarp_lists_dict, prepare_to_swarp, run_swarp

In [ ]:
# set the directory folders
swarp_path = "/opt/local/bin/swarp"
config_dir = os.getcwd() + '/' + '../config/'

config = pycavi_tools.read_pycavi_config(config_dir)
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
swarp_dir_name = config['Directories']['swarp_dir_name'] 
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_swarp.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# all this part is to select all the .head files and grouping accordling to observation and filter
# the grouped images are stored in the swarp_list dictionary, where the key correspond to the combination
# of observation plus filter, and the element to corresponding list of images
# print(glob.glob(reduced_science_dir+"/*/*/*/*_pre/*_wcs.head"))
#
# To select all data partitions leave the list below empty
filters = set(['BB_B_99_ESO842', 'BB_V_89_ESO843', 'g_SDSS', 'r_SDSS', 'NB_659'])
# filters = ["NB_659"]

# We clean the list of bad observations - this should be done by hand
reject_observations=[]


# We first list the standard stars fields data partitions
swarp_lists_std = get_swarp_lists_dict(reduced_std_dir, reject_observations, filters)
std_observation_tags = list(set(swarp_lists_std.keys()))
std_observation_tags.sort()                # Needed to ensure same index in every run
for i in range(len(std_observation_tags)):
    logging.info("index: {} observation_tag: {}".format(i, std_observation_tags[i]))

# We next list the science observation fields data partitions
swarp_lists_sci = get_swarp_lists_dict(reduced_science_dir, reject_observations, filters)
sci_observation_tags = list(set(swarp_lists_sci.keys()))    
sci_observation_tags.sort()                # Needed to ensure same index in every run
for i in range(len(sci_observation_tags)):
    logging.info("index: {} observation_tag: {}".format(i, sci_observation_tags[i]))
    

In [ ]:
# We first select what to swarp
# If only a few data partitions write the indices to them below.
# The loop below will skip every index but those listed.
# If the list idx_selected is empty, then all data partitions will be swarped.
#
obs_tags = sci_observation_tags
swarp_lists = swarp_lists_sci
reduced_dir = reduced_science_dir
idx_selected = []

#
# updates some SWarp parameters 
# ---------- General processing parameters -----------------------
special_pars = {}

# coordinate type
special_pars["CELESTIAL_TYPE"] = "NATIVE"    #  "NATIVE, PIXEL, EQUATORIAL,GALACTIC,ECLIPTIC, or SUPERGALACTIC"
special_pars["PROJECTION_TYPE"] = "TAN"      # Any WCS projection code or NONE
                                             # TAN: Distorted tangential (default)
                                             # ------- Flux preserving that use PVi_j coeffs (See Table 2 of Swarp manual)
                                             # CEA: Cylindrical equal-area
                                             # COE: Conic equal area
                                             # BON: Bonne’s equal-area
                                             # and many others


special_pars["PROJECTION_ERR"]  = 0.001      # Maximum projection error (in output pixels), or 0 for no approximation

# processing parameters
special_pars["SATLEV_KEYWORD"] = "SATLEVEL"  # FITS key containing saturation level
special_pars["RESAMPLE_DIR"] = "./resamp"
special_pars["VMEM_DIR"] = "./swap"

# keywords to copy
special_pars["COPY_KEYWORDS"] = "OBJECT,RA,DEC,OBJECT,PIXSCALE,ROT-RTA,INSTRUME,FILTER,TELESCOPE"



# ----------- Parameters governing sky subtraction ----------------
special_pars["BACK_SIZE"] = 128            # Background mesh size (pixels) all or for each image [128]
special_pars["BACK_FILTERSIZE"] = 3        # Background map filter range (meshes) (all or for each image) [3]
special_pars["BACK_FILTTHRESH"] = 0.0      # Threshold above which the background-map filter operates

# ------------Parameters governing Co-addition --------------------------
# To combine in one go choose the values below. This will remove individual files.
#
special_pars["COMBINE"] = "Y"              # Combine resampled images (Y/N)? If "N" automatically sets 
                                           # "DELETE_TMPFILES" to "N"
                                           # In a second pass "RESAMPLE" should be set to "N", just to combine
                                           # frames. According to manual this works only if SAME_CRVAL and CDELT
                                           # which sounds a bit outdated so this should be investigated (no CDELT anymore).
special_pars["COMBINE_TYPE"] = "MEDIAN"    # "MEDIAN,AVERAGE,MIN,MAX,WEIGHTED,CHI2 or SUM

# We could also keep the individual swarped files by setting the parameter below to "N"
special_pars["DELETE_TMPFILES"] = "N"

for i in range(len(obs_tags)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
        
    observation_tag = obs_tags[i]
    logging.info("SWarping observation {}, filter {}".format(observation_tag.split("/")[0],
                                                             observation_tag.split("/")[1]))
    #print(swarp_lists[observation_tag])

    observation = swarp_lists[observation_tag][0].split("/")[-5]
    filter_ = swarp_lists[observation_tag][0].split("/")[-4]
    frames_list_name = observation + "_" + filter_ + ".list"
    special_pars["IMAGEOUT_NAME"] = frames_list_name[:-5] + ".coadd.fits"
    special_pars["WEIGHTOUT_NAME"] = frames_list_name[:-5] + ".weight.fits"    
    if "NB_659" in filter_:
        special_pars["SUBTRACT_BACK"] = "N"
    else:
        special_pars["SUBTRACT_BACK"] = "Y"
    swarp_dir_path = reduced_dir + observation + '/' + swarp_dir_name

    # For debugging you might want to edit the frame list and run with 
    # reuse_frame_list = True.
    reuse_frame_list = True
    if frames_list_name is not None:
        if reuse_frame_list and os.path.exists(swarp_dir_path+'/'+frames_list_name):
            prepare_to_swarp(swarp_lists[observation_tag], swarp_dir_path, None, special_pars)
            run_swarp(swarp_dir_path, frames_list_name, special_pars=special_pars, path=swarp_path)
        else:
            frames_list_name = prepare_to_swarp(swarp_lists[observation_tag], swarp_dir_path, frames_list_name, special_pars)
            run_swarp(swarp_dir_path, frames_list_name, special_pars=special_pars, path=swarp_path)
    else:
        logging.info("Could not run swarp: frame_list_name is None")
    
logging.info('-------------- STD SWARP DONE ---------------') 